In [39]:
import openai
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langsmith import Client
import json
load_dotenv()


True

In [2]:
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
collection_name = os.getenv("QDRANT_COLLECTION_NAME")


In [52]:
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY,timeout=60)


## Download all data From Qdrant

In [4]:
all_points=qdrant_client.scroll(
    collection_name=collection_name,
    limit=300,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [5]:
all_points

([Record(id=1, payload={'description': 'Gvoears Replacement Ear Pads Cushions for Skullcandy Crusher Wireless, Crusher ANC/EVO, Hesh ANC/EVO, Hesh 3 Wireless, Also Fit for Skullcandy Venue Wireless ANC Headphone with Duable Leather Fabric【Perfect Compatibility】These ear pads are made specifically to fit the Skullcandy Hesh 3, Evo, and Crusher series of Headphones. You can comfortably upgrade your ear cushion for skullcandy crusher wireless over-ear headphones, hesh 3 wireless over-ear headphone, crusher evo/anc wireless over-ear headphones, hesh evo/anc wireless over-ear headphones, for skullcandy venue wireless over-ear headphone.【Durable&Comfortable】Crusher wireless replacement pads are designed with soft leather fabric to make your ears do not pain when you wear them for a long time. Durable design make Your headphone can be used for a long time. keep them comfortable and take care of your ears.【Noise Isolation】Earmuffs for skullcandy wireless constructed of premium memory foam, thi

In [6]:
all_points[0][0].payload

{'description': 'Gvoears Replacement Ear Pads Cushions for Skullcandy Crusher Wireless, Crusher ANC/EVO, Hesh ANC/EVO, Hesh 3 Wireless, Also Fit for Skullcandy Venue Wireless ANC Headphone with Duable Leather Fabric【Perfect Compatibility】These ear pads are made specifically to fit the Skullcandy Hesh 3, Evo, and Crusher series of Headphones. You can comfortably upgrade your ear cushion for skullcandy crusher wireless over-ear headphones, hesh 3 wireless over-ear headphone, crusher evo/anc wireless over-ear headphones, hesh evo/anc wireless over-ear headphones, for skullcandy venue wireless over-ear headphone.【Durable&Comfortable】Crusher wireless replacement pads are designed with soft leather fabric to make your ears do not pain when you wear them for a long time. Durable design make Your headphone can be used for a long time. keep them comfortable and take care of your ears.【Noise Isolation】Earmuffs for skullcandy wireless constructed of premium memory foam, this headphone ear covers 

In [7]:
all_context=[{"id":data.payload['parent_asin'],"text":data.payload["description"]} for data in all_points[0] ]

In [8]:
all_context

[{'id': 'B0BTBHYWY4',
  'text': 'Gvoears Replacement Ear Pads Cushions for Skullcandy Crusher Wireless, Crusher ANC/EVO, Hesh ANC/EVO, Hesh 3 Wireless, Also Fit for Skullcandy Venue Wireless ANC Headphone with Duable Leather Fabric【Perfect Compatibility】These ear pads are made specifically to fit the Skullcandy Hesh 3, Evo, and Crusher series of Headphones. You can comfortably upgrade your ear cushion for skullcandy crusher wireless over-ear headphones, hesh 3 wireless over-ear headphone, crusher evo/anc wireless over-ear headphones, hesh evo/anc wireless over-ear headphones, for skullcandy venue wireless over-ear headphone.【Durable&Comfortable】Crusher wireless replacement pads are designed with soft leather fabric to make your ears do not pain when you wear them for a long time. Durable design make Your headphone can be used for a long time. keep them comfortable and take care of your ears.【Noise Isolation】Earmuffs for skullcandy wireless constructed of premium memory foam, this headp

### Render a prompt to generate synthetic Eval Reference dataset

In [25]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "string",
                "description": "Suggested question",
            },
            "chunck_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the questions",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context",
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks",
            },
        },
    },
}

SYSTEM_PROMPT=f"""
I am building a RAG application. I have a collection of 300 chunks of text.

The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.

I will provide all of the available products to you with IDs for each chunk.

I want you to come up with 100 questions to which the answers could be grounded in the chunk context.

The questions should imitate a potential real user of this RAG system.

As an output I need you to provide:
- The list of questions.
- The IDs of the chunks that could be used to answer them.
- An example answer to each question, grounded in the provided context.
- The reasoning why you chose those chunks to answer the question.

Constraints:
- 10 questions must require multiple chunks to answer.
- 15 questions must require only a single chunk to answer.
- 5 questions should intentionally **not** be answerable using the available chunks.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

The output must be valid JSON and strictly follow the provided schema so that I can parse it programmatically.


"""

USER_PROMPT=f"""
here is the List of Chunks each list element is a dictionry with id and text

{all_context}

"""

In [26]:
print(SYSTEM_PROMPT)


I am building a RAG application. I have a collection of 300 chunks of text.

The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.

I will provide all of the available products to you with IDs for each chunk.

I want you to come up with 100 questions to which the answers could be grounded in the chunk context.

The questions should imitate a potential real user of this RAG system.

As an output I need you to provide:
- The list of questions.
- The IDs of the chunks that could be used to answer them.
- An example answer to each question, grounded in the provided context.
- The reasoning why you chose those chunks to answer the question.

Constraints:
- 10 questions must require multiple chunks to answer.
- 15 questions must require only a single chunk to answer.
- 5 questions should intentionally **not** be answerable using the available chunks.

<OUTPUT JSON SCHEMA>
{
  "type": "array",
  "items": {
    "type"

In [27]:

from pydantic import BaseModel, Field
from typing import List, Literal

class RAGQuestion(BaseModel):
    question: str = Field(description="Suggested user question")
    chunk_ids: List[str] = Field(
        description="IDs of chunks used to answer the question. Empty for unanswerable questions."
    )
    answer_example: str = Field(
        description="Example answer grounded in context. Empty if unanswerable."
    )
    reasoning: str = Field(
        description="Why these chunks can or cannot answer the question"
    )
    answerable: bool = Field(
        description="Whether the question can be answered from the provided chunks"
    )
    chunk_requirement: Literal["single", "multiple", "none"] = Field(
        description="Single chunk, multiple chunks, or no available chunks"
    )

class RAGEvaluationDataset(BaseModel):
    questions: List[RAGQuestion]


In [28]:

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    response_model=RAGEvaluationDataset,
    reasoning_effort="minimal"
)

print(response)



questions=[RAGQuestion(question='Do you have replacement ear pads for Skullcandy Crusher Wireless headphones?', chunk_ids=['B0BTBHYWY4'], answer_example='Yes — we have Gvoears Replacement Ear Pads designed specifically for Skullcandy Crusher Wireless, Crusher ANC/EVO, Hesh ANC/EVO, Hesh 3 Wireless and Skullcandy Venue Wireless ANC. They come as 2 pads (3.74" x 3.15" x 0.98"), include 4 wipe pads and 2 cable velcro straps, and feature memory foam, leather fabric and easy snap-in installation.', reasoning='Chunk B0BTBHYWY4 is the product listing for Gvoears replacement ear pads; it contains compatibility, dimensions, contents, and features needed to answer.', answerable=True, chunk_requirement='single'), RAGQuestion(question='Which Ethernet splitter can convert one RJ45 socket into two and does it allow two devices to use internet simultaneously?', chunk_ids=['B0BWXS9QVY'], answer_example='The NOBVEQ RJ45 Ethernet Splitter (1 male to 2 female) converts a single RJ45 socket into two. It s

In [29]:

for q in response.questions:
    print("Question:", q.question)
    print("Chunk IDs:", q.chunk_ids)
    print("Answer:", q.answer_example)
    print("Reasoning:", q.reasoning)
    print("Answerable:", q.answerable)
    print("-" * 50)


Question: Do you have replacement ear pads for Skullcandy Crusher Wireless headphones?
Chunk IDs: ['B0BTBHYWY4']
Answer: Yes — we have Gvoears Replacement Ear Pads designed specifically for Skullcandy Crusher Wireless, Crusher ANC/EVO, Hesh ANC/EVO, Hesh 3 Wireless and Skullcandy Venue Wireless ANC. They come as 2 pads (3.74" x 3.15" x 0.98"), include 4 wipe pads and 2 cable velcro straps, and feature memory foam, leather fabric and easy snap-in installation.
Reasoning: Chunk B0BTBHYWY4 is the product listing for Gvoears replacement ear pads; it contains compatibility, dimensions, contents, and features needed to answer.
Answerable: True
--------------------------------------------------
Question: Which Ethernet splitter can convert one RJ45 socket into two and does it allow two devices to use internet simultaneously?
Chunk IDs: ['B0BWXS9QVY']
Answer: The NOBVEQ RJ45 Ethernet Splitter (1 male to 2 female) converts a single RJ45 socket into two. It splits pins into two sets (1236/4578) 

In [30]:
len(response.questions)

31

In [55]:
# Create a local dictionary mapping ID (parent_asin) -> description
id_to_desc = {item["id"]: item["text"] for item in all_context}

# Instant O(1) lookup — 0 network requests, no timeouts
def get_description(parent_asin: str) -> str:
    return id_to_desc.get(parent_asin, "")

In [47]:
langsmith_client=Client(api_key=os.environ['LANGCHAIN_API_KEY'])

In [41]:
dataset_name="shopmart_eval_dataset"
dataset=langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description="Datasets for evaluating Rag pipeline"
)

In [57]:
for item in response.questions:
    langsmith_client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item.question},
        outputs={
            "ground_truth": item.answer_example,
            "reference_context_ids": item.chunk_ids,
            "reference_description": [
                get_description(id) for id in item.chunk_ids
            ],
            "parent_asin": item.chunk_ids
        }
    )